# AG_PRAXIS NB06c — Do Conformal Prediction Sets Carry Information Here

This is a probe and not a stage of the pipeline. It reads the probability matrices that
were saved earlier, wraps them in a class-conditional conformal procedure, and looks at
one quantity: how many classes end up in each test window's prediction set. No model is
built, nothing is written to disk, and no number produced here goes into the results
ledger. The question it answers is whether the set sizes carry information at all, which
is what decides whether a full conformal analysis is worth building.

A prediction set is the set of classes the procedure is not willing to rule out for a
given window, and its size is a direct statement about ambiguity. If the model has a
clean picture of a class, most windows of that class should come back with one class in
the set. If a class sits on top of others in feature space, its windows should come back
with several. That reading holds whether or not the coverage guarantee behind the
procedure is actually met, which matters here because two of the nineteen classes leave
so few validation windows that a guarantee resting on them would not mean much. Set sizes
are being read as a measure of confusion, not as evidence of a guarantee.

Three classes are singled out before anything runs, so the reading is not chosen after
seeing the numbers. DDoS-SYN is a volumetric class the model scores well. Recon-VulScan
is a low-rate class it scores badly. MQTT-DDoS-Publish_Flood is the class the pairwise
feature separability measured earlier in this repository put among the least separable,
and it is the one this probe is really about: if a method that never sees any of that
analysis independently returns large ambiguous sets for it, two different measurements
are pointing at the same thing. If the sets come back the same size for all three, the
sizes are measuring nothing and there is no point building anything on them.


The arrays are on Drive and the code is in the repository, so the first block mounts one
and clones the other, and records the commit it is running from.


In [1]:
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"
NOTEBOOK = "AG_PRAXIS_NB06c_cp_feasibility.ipynb"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")


Mounted at /content/drive
colab     : True
repo root : /content/repo
git sha   : 73f0ea3 on main
run date  : 2026-08-10


Everything this notebook needs was written by the run that saved the scores: the two
probability matrices, the labels beside them, and a small file recording the class order
those columns are in. The class order is read from that file rather than retyped, because
a column mistaken for its neighbour would produce a full set of plausible numbers and no
error.

One other file is read, and only for context at the end: the metrics of the model run the
probabilities came from, which holds the per-class F1 already measured on this same test
partition. Set size and F1 are two different measurements of the same thing, so putting
them side by side is the check on whether set size is measuring difficulty or noise.

The two coverage targets are fixed here. 0.90 is the one reported throughout, and 0.80 is
carried alongside it to see how much of the picture depends on where that dial is set.
Nothing is written anywhere, so no output directory is defined.


In [2]:
import json

import numpy as np
import pandas as pd

from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
ARTIFACTS = Path(CFG["paths"]["artifacts"])
SCORES_DIR = (
    (ARTIFACTS / "NB06b_cp_scores") if IN_COLAB else (REPO_ROOT / "results" / "NB06b_cp_scores")
)

COVERAGES = (0.90, 0.80)
PRIMARY = 0.90
FOCUS = ("DDoS-SYN", "Recon-VulScan", "MQTT-DDoS-Publish_Flood")
SMALL_CALIBRATION = 100

SCORES_PATH = SCORES_DIR / "scores.json"
if not SCORES_PATH.exists():
    raise FileNotFoundError(
        f"{SCORES_PATH} is missing. This notebook reads the probability matrices that the "
        "score-saving run wrote and does not produce them. Check that Drive is mounted."
    )

SCORES = json.loads(SCORES_PATH.read_text())
CLASSES = list(SCORES["labels"])
N_CLASSES = len(CLASSES)
INDEX_OF = {name: i for i, name in enumerate(CLASSES)}

missing = [name for name in FOCUS if name not in INDEX_OF]
if missing:
    raise KeyError(f"the classes this probe is about are not in the label list: {missing}")


def optional_path(candidates):
    return next((Path(p) for p in candidates if Path(p).exists()), None)


MODEL_RUN = SCORES.get("model_run_id", "sequence_cnn_lstm_19class")
NB06_METRICS_PATH = optional_path(
    [
        REPO_ROOT / "data" / "processed" / "NB06" / "metrics.json",
        ARTIFACTS / "NB06" / MODEL_RUN / "metrics.json",
    ]
)

PER_CLASS_F1 = None
if NB06_METRICS_PATH is not None:
    NB06_METRICS = json.loads(NB06_METRICS_PATH.read_text())
    if sorted(NB06_METRICS["labels"]) == sorted(CLASSES):
        PER_CLASS_F1 = {c: float(NB06_METRICS["per_class_f1"][c]) for c in CLASSES}

f1_source = (
    f"read from {NB06_METRICS_PATH}"
    if PER_CLASS_F1
    else "not found, the difficulty comparison at the end will be skipped"
)

print(f"scores from : {SCORES_DIR}")
print(f"written by  : {SCORES.get('written_by')} on {SCORES.get('run_date')} "
      f"at {SCORES.get('git_sha')}")
print(f"model run   : {MODEL_RUN}")
print(f"classes     : {N_CLASSES}")
print(f"per-class F1: {f1_source}")
print(f"coverage    : {PRIMARY} reported throughout, {COVERAGES} computed")


scores from : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB06b_cp_scores
written by  : AG_PRAXIS_NB06b_cp_scores.ipynb on 2026-08-10 at 34f8ffd
model run   : sequence_cnn_lstm_19class
classes     : 19
per-class F1: read from /content/repo/data/processed/NB06/metrics.json
coverage    : 0.9 reported throughout, (0.9, 0.8) computed


Nothing here is random. No model is built, no sampling happens, and reading an array and
sorting it gives the same answer every time. The seed is set anyway, because the rule in
this project is that it is set before anything else in the session, and a rule followed
only when it appears to matter is not being followed.


In [3]:
import random

random.seed(SEED)
np.random.seed(SEED)

print(f"seed {SEED} set")


seed 42 set


Now the arrays. The validation matrix is the calibration set, which is the partition the
model was neither trained on nor tuned on, and the test matrix is what the calibrated
procedure is then applied to. Each is checked as it arrives: one row per label, one column
per class, every row summing to one, and the per-class counts equal to the counts the
score file recorded. A matrix whose rows do not sum to one is not a set of probabilities
and every threshold taken from it would be meaningless, and per-class counts that do not
match would mean the labels and the rows are not the pairs they are being read as.


In [4]:
def read_partition(name):
    """One partition's probabilities and labels, checked against the score file."""
    probs_path = SCORES_DIR / f"probs_{name}.npy"
    labels_path = SCORES_DIR / f"y_true_{name}.npy"
    for path in (probs_path, labels_path):
        if not path.exists():
            raise FileNotFoundError(f"{path} is missing")

    probs = np.load(probs_path).astype("float64")
    y = np.load(labels_path).astype("int64")

    if probs.ndim != 2 or probs.shape[1] != N_CLASSES:
        raise ValueError(f"{probs_path.name} is {probs.shape}, not (n, {N_CLASSES})")
    if len(y) != len(probs):
        raise ValueError(f"{len(y)} labels against {len(probs)} rows in {name}")
    if not np.isfinite(probs).all():
        raise ValueError(f"{probs_path.name} holds a value that is not finite")
    if y.min() < 0 or y.max() >= N_CLASSES:
        raise ValueError(f"{labels_path.name} holds a label outside 0..{N_CLASSES - 1}")

    row_sums = probs.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-4):
        raise ValueError(
            f"{probs_path.name} rows sum to between {row_sums.min():.6f} and "
            f"{row_sums.max():.6f}, so they are not probability distributions"
        )

    counted = {c: int(n) for c, n in zip(CLASSES, np.bincount(y, minlength=N_CLASSES))}
    recorded = SCORES["partitions"][name]["by_class"]
    if counted != {k: int(v) for k, v in recorded.items()}:
        raise ValueError(f"{name} per-class counts differ from the ones scores.json records")

    print(f"  {probs_path.name:<16} {str(probs.shape):>16}  rows sum to "
          f"[{row_sums.min():.6f}, {row_sums.max():.6f}]  {len(y):,} labels")
    return probs, y


print("reading")
PROBS_VAL, Y_VAL = read_partition("val")
PROBS_TEST, Y_TEST = read_partition("test")

VAL_N = np.bincount(Y_VAL, minlength=N_CLASSES)
TEST_N = np.bincount(Y_TEST, minlength=N_CLASSES)

print()
print(f"calibration set {len(Y_VAL):,} windows, evaluation set {len(Y_TEST):,} windows")
print(f"smallest calibration classes: " + ", ".join(
    f"{CLASSES[c]} {VAL_N[c]}" for c in np.argsort(VAL_N)[:3]
))


reading
  probs_val.npy         (52637, 19)  rows sum to [1.000000, 1.000000]  52,637 labels
  probs_test.npy        (49159, 19)  rows sum to [1.000000, 1.000000]  49,159 labels

calibration set 52,637 windows, evaluation set 49,159 windows
smallest calibration classes: Recon-Ping_Sweep 2, Recon-VulScan 18, MQTT-Malformed_Data 38


The nonconformity score is the plain one: for a window and a candidate class, the score is
one minus the probability the model gave that class. A low score means the model was
comfortable with that class for that window, a high score means it was not.

The calibration is done per class rather than over the pool, which is the Mondrian form of
split conformal. Every class gets its own threshold, taken from the scores its own
validation windows produced on their own true label. The reason to do it this way here is
that the classes are wildly unequal in size and difficulty, and a single pooled threshold
would be set almost entirely by the large easy classes and would then be applied to the
small hard ones, where it would say nothing about them.

The threshold for a class is not simply the empirical quantile at the target coverage but
the score sitting at rank ceil((n + 1) x coverage) among that class's n calibration
scores. The plus one is what makes the finite-sample statement work: with n points the
rank has to be pushed slightly past the naive position to account for the unseen point
being scored. That correction is invisible when n is in the thousands and decisive when n
is small. When the required rank exceeds n, no finite threshold exists at that coverage
and the class is admitted to every set, which is the honest answer rather than an
extrapolation.

That is where the small classes have to be flagged. A threshold taken at or near the top
rank of eighteen points is one order statistic of eighteen numbers, and one unusual
validation window moves it a long way. The table below prints the count and the rank used
for every class so this is visible rather than buried. Nothing fails on these classes and
they are carried through the rest of the notebook, but any reading of their thresholds
carries that caveat with it.


In [5]:
def conformal_thresholds(probs, y, coverage):
    """Per-class threshold on s = 1 - p_true, at the conformal rank for this coverage."""
    thresholds = np.full(N_CLASSES, np.inf)
    rows = []
    for c, name in enumerate(CLASSES):
        scores = np.sort(1.0 - probs[y == c, c])
        n = int(scores.size)
        rank = int(np.ceil((n + 1) * coverage))
        if n == 0:
            note = "no calibration windows"
        elif rank > n:
            note = f"rank {rank} needed, only {n} points: no finite threshold, admits every window"
        else:
            thresholds[c] = float(scores[rank - 1])
            if rank == n:
                note = f"the largest of {n} points"
            elif n < SMALL_CALIBRATION:
                note = f"only {n} points"
            else:
                note = ""
        rows.append(
            {
                "class": name,
                "cal_n": n,
                "rank_used": rank,
                "threshold": thresholds[c],
                "p_true_needed": 1.0 - thresholds[c],
                "cal_score_median": float(np.median(scores)) if n else float("nan"),
                "note": note,
            }
        )
    table = pd.DataFrame(rows)
    for column in ("cal_n", "rank_used", "threshold", "p_true_needed", "cal_score_median"):
        if not pd.api.types.is_numeric_dtype(table[column]):
            raise TypeError(f"{column} came out as {table[column].dtype}, not numeric")
    return thresholds, table


THRESHOLDS, THRESHOLD_TABLE = {}, {}
for coverage in COVERAGES:
    THRESHOLDS[coverage], THRESHOLD_TABLE[coverage] = conformal_thresholds(
        PROBS_VAL, Y_VAL, coverage
    )

shown = THRESHOLD_TABLE[PRIMARY].sort_values("cal_n")
print(f"per-class thresholds at coverage {PRIMARY}, smallest calibration class first")
print(shown.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

flagged = shown[shown["note"] != ""]
print()
print(f"{len(flagged)} of {N_CLASSES} classes calibrate on fewer than {SMALL_CALIBRATION} "
      f"points or on their own largest score:")
for _, row in flagged.iterrows():
    print(f"  {row['class']:<26} n={int(row.cal_n):<6} rank={int(row.rank_used):<6} {row['note']}")


per-class thresholds at coverage 0.9, smallest calibration class first
                  class  cal_n  rank_used  threshold  p_true_needed  cal_score_median                                                                   note
       Recon-Ping_Sweep      2          3        inf           -inf          0.872761 rank 3 needed, only 2 points: no finite threshold, admits every window
          Recon-VulScan     18         18   0.999970       0.000030          0.999175                                               the largest of 18 points
    MQTT-Malformed_Data     38         36   0.926158       0.073842          0.230407                                                         only 38 points
 MQTT-DoS-Connect_Flood     92         84   0.000445       0.999555          0.000134                                                         only 92 points
               Spoofing    105         96   0.975642       0.024358          0.705683                                                           

A window's prediction set is now every class whose score for that window falls at or below
that class's threshold, which is the same as saying the model gave the class at least the
probability that class's threshold demands. Each class is judged against its own bar, so a
class calibrated loosely will appear in many sets and a class calibrated tightly in few.

The size of the set is the output this probe is about, and it needs nothing from the
coverage argument to be meaningful. It is a count of the classes the procedure could not
rule out. Sets of size zero are possible and are counted separately: they are windows
where every class fell below its own bar, which is the procedure saying it recognises
nothing here rather than saying nothing is there.

One thing has to be separated out before any of these counts are read. A class with no
finite threshold is admitted to every set by construction, so it adds exactly one to the
size of every window in the dataset and tells us nothing about any of them. That is an
artefact of having too few calibration points for that class, not a statement about the
window it appears in. The count is therefore reported twice: once over all classes, and
once over only the classes that have a finite threshold. The second is the one to compare
across classes, because the constant is gone from it.


In [6]:
def prediction_sets(probs, thresholds):
    """Boolean membership matrix: class c is in row i's set when 1 - p[i, c] <= threshold[c]."""
    return (1.0 - probs) <= thresholds[None, :]


MEMBER, SIZES, SIZES_FINITE, COVERED, FINITE, ALWAYS_IN = {}, {}, {}, {}, {}, {}
for coverage in COVERAGES:
    MEMBER[coverage] = prediction_sets(PROBS_TEST, THRESHOLDS[coverage])
    SIZES[coverage] = MEMBER[coverage].sum(axis=1).astype("int64")
    FINITE[coverage] = np.isfinite(THRESHOLDS[coverage])
    SIZES_FINITE[coverage] = MEMBER[coverage][:, FINITE[coverage]].sum(axis=1).astype("int64")
    COVERED[coverage] = MEMBER[coverage][np.arange(len(Y_TEST)), Y_TEST]
    ALWAYS_IN[coverage] = [CLASSES[c] for c in np.flatnonzero(~FINITE[coverage])]

for coverage in COVERAGES:
    sizes, sizes_finite = SIZES[coverage], SIZES_FINITE[coverage]
    admitted = ALWAYS_IN[coverage]
    print(f"coverage {coverage}: set size over all {len(sizes):,} test windows, "
          f"mean {sizes.mean():.3f}, median {int(np.median(sizes))}, max {int(sizes.max())}")
    print(f"{'':<14}empty {int((sizes == 0).sum()):,} "
          f"({(sizes == 0).mean():.4f}), singleton {int((sizes == 1).sum()):,} "
          f"({(sizes == 1).mean():.4f}), all {N_CLASSES} classes "
          f"{int((sizes == N_CLASSES).sum()):,} ({(sizes == N_CLASSES).mean():.4f})")
    if admitted:
        print(f"{'':<14}{len(admitted)} class(es) have no finite threshold and sit in every "
              f"set: {', '.join(admitted)}")
        print(f"{'':<14}over the {int(FINITE[coverage].sum())} classes that do have one, "
              f"mean {sizes_finite.mean():.3f}, median {int(np.median(sizes_finite))}, "
              f"singleton {(sizes_finite == 1).mean():.4f}")
    else:
        print(f"{'':<14}every class has a finite threshold, so no constant is being added")
    print()


coverage 0.9: set size over all 49,159 test windows, mean 2.895, median 3, max 7
              empty 0 (0.0000), singleton 437 (0.0089), all 19 classes 0 (0.0000)
              1 class(es) have no finite threshold and sit in every set: Recon-Ping_Sweep
              over the 18 classes that do have one, mean 1.895, median 2, singleton 0.3499

coverage 0.8: set size over all 49,159 test windows, mean 2.543, median 2, max 6
              empty 0 (0.0000), singleton 1,529 (0.0311), all 19 classes 0 (0.0000)
              1 class(es) have no finite threshold and sit in every set: Recon-Ping_Sweep
              over the 18 classes that do have one, mean 1.543, median 1, singleton 0.4880



The three classes chosen at the start, one at a time. For each, the set sizes of the test
windows whose true class is that one: the mean, the middle, and then the whole distribution
of sizes rather than a summary of it, since a mean of two could be every window at two or
half at one and half at three and those mean different things.


In [7]:
def size_distribution(sizes, top=8):
    """The distribution of set sizes as counts and shares, largest group first."""
    values, counts = np.unique(sizes, return_counts=True)
    frame = pd.DataFrame({"set_size": values, "windows": counts})
    frame["share"] = frame["windows"] / frame["windows"].sum()
    for column in ("set_size", "windows", "share"):
        if not pd.api.types.is_numeric_dtype(frame[column]):
            raise TypeError(f"{column} came out as {frame[column].dtype}, not numeric")
    return frame.sort_values("windows", ascending=False).head(top)


for name in FOCUS:
    c = INDEX_OF[name]
    mask = Y_TEST == c
    sizes = SIZES[PRIMARY][mask]
    sizes_finite = SIZES_FINITE[PRIMARY][mask]
    sizes_80 = SIZES[0.80][mask]

    print("=" * 78)
    print(f"{name}   {int(mask.sum()):,} test windows, calibrated on {int(VAL_N[c])}")
    if PER_CLASS_F1 is not None:
        print(f"F1 already measured on this partition: {PER_CLASS_F1[name]:.4f}")
    print(f"threshold {THRESHOLDS[PRIMARY][c]:.6f}, so a window joins this class's set at "
          f"p >= {1 - THRESHOLDS[PRIMARY][c]:.6f}")
    print()
    print(f"set size at coverage {PRIMARY}: mean {sizes.mean():.3f}, median "
          f"{int(np.median(sizes))}, 90th percentile {int(np.percentile(sizes, 90))}, "
          f"max {int(sizes.max())}")
    if ALWAYS_IN[PRIMARY]:
        print(f"  over the finitely thresholded classes only: mean {sizes_finite.mean():.3f}, "
              f"median {int(np.median(sizes_finite))}")
    print(f"set size at coverage 0.80: mean {sizes_80.mean():.3f}, median "
          f"{int(np.median(sizes_80))}")
    print(f"singletons {(sizes == 1).mean():.4f}, empty {(sizes == 0).mean():.4f}, "
          f"five or more {(sizes >= 5).mean():.4f}")
    print()
    print(f"the distribution at coverage {PRIMARY}")
    print(size_distribution(sizes).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print()


DDoS-SYN   6,894 test windows, calibrated on 7874
F1 already measured on this partition: 0.8926
threshold 0.763614, so a window joins this class's set at p >= 0.236386

set size at coverage 0.9: mean 2.713, median 3, 90th percentile 4, max 6
  over the finitely thresholded classes only: mean 1.713, median 2
set size at coverage 0.80: mean 2.444, median 2
singletons 0.0000, empty 0.0000, five or more 0.0044

the distribution at coverage 0.9
 set_size  windows  share
        3     3005 0.4359
        2     2951 0.4281
        4      908 0.1317
        5       26 0.0038
        6        4 0.0006

Recon-VulScan   18 test windows, calibrated on 18
F1 already measured on this partition: 0.5385
threshold 0.999970, so a window joins this class's set at p >= 0.000030

set size at coverage 0.9: mean 2.833, median 3, 90th percentile 3, max 5
  over the finitely thresholded classes only: mean 1.833, median 2
set size at coverage 0.80: mean 2.389, median 2
singletons 0.0000, empty 0.0000, five or m

The question those three were chosen to answer. If set size is measuring ambiguity, the
class the model handles well should come back small and the classes it handles badly
should come back large, and the ordering should hold without anyone arranging it. If the
three come out at roughly the same size, the sets are being driven by something other than
how hard the classes are, and the rest of this notebook is not worth reading.


In [8]:
easy, hard, target = FOCUS
mean_size = {
    name: float(SIZES[PRIMARY][Y_TEST == INDEX_OF[name]].mean()) for name in FOCUS
}
mean_size_finite = {
    name: float(SIZES_FINITE[PRIMARY][Y_TEST == INDEX_OF[name]].mean()) for name in FOCUS
}

print(f"mean prediction-set size at coverage {PRIMARY}, all classes then finite only")
for name in FOCUS:
    print(f"  {name:<28} {mean_size[name]:>7.3f}   {mean_size_finite[name]:>7.3f}")
print()

hard_larger = mean_size[hard] > mean_size[easy]
target_larger = mean_size[target] > mean_size[easy]
spread = max(mean_size.values()) - min(mean_size.values())

print(f"{hard} against {easy}: "
      f"{'larger' if hard_larger else 'not larger'} "
      f"({mean_size[hard]:.3f} against {mean_size[easy]:.3f}, "
      f"{mean_size[hard] - mean_size[easy]:+.3f})")
print(f"{target} against {easy}: "
      f"{'larger' if target_larger else 'not larger'} "
      f"({mean_size[target]:.3f} against {mean_size[easy]:.3f}, "
      f"{mean_size[target] - mean_size[easy]:+.3f})")
print(f"spread across the three: {spread:.3f} classes")
print()
print("the expected ordering holds on both counts"
      if hard_larger and target_larger
      else "the expected ordering does not hold on both counts")


mean prediction-set size at coverage 0.9, all classes then finite only
  DDoS-SYN                       2.713     1.713
  Recon-VulScan                  2.833     1.833
  MQTT-DDoS-Publish_Flood        2.535     1.535

Recon-VulScan against DDoS-SYN: larger (2.833 against 2.713, +0.120)
MQTT-DDoS-Publish_Flood against DDoS-SYN: not larger (2.535 against 2.713, -0.178)
spread across the three: 0.298 classes

the expected ordering does not hold on both counts


Then the class this probe is really about, on its own. What is wanted is not only whether
its sets are large but what is in them, because a class whose windows come back with four
classes in the set is being confused with three specific others, and which three is a
statement about where it sits in feature space. If the classes it is mixed with are the
ones the separability analysis put it next to, that is two measurements agreeing from
different directions. If its sets are singletons, the conformal view does not corroborate
that finding and should be reported as not corroborating it.


In [9]:
c = INDEX_OF[target]
mask = Y_TEST == c
member = MEMBER[PRIMARY][mask]
sizes = SIZES[PRIMARY][mask]

print(f"{target}, {int(mask.sum()):,} test windows")
print(f"mean set size {sizes.mean():.3f}, median {int(np.median(sizes))}, "
      f"singletons {(sizes == 1).mean():.4f}, empty {(sizes == 0).mean():.4f}")
print(f"over the finitely thresholded classes only, mean "
      f"{SIZES_FINITE[PRIMARY][mask].mean():.3f}")
print(f"the true class is in its own set on {member[:, c].mean():.4f} of them")
print()

company = pd.DataFrame(
    {
        "class_in_set": CLASSES,
        "share_of_windows": member.mean(axis=0),
        "is_the_true_class": [name == target for name in CLASSES],
        "has_a_finite_threshold": FINITE[PRIMARY],
    }
).sort_values("share_of_windows", ascending=False)

if not pd.api.types.is_numeric_dtype(company["share_of_windows"]):
    raise TypeError("share_of_windows came out non-numeric")

print(f"which classes appear in the sets of {target} windows")
print(company[company["share_of_windows"] > 0]
      .head(10)
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))


MQTT-DDoS-Publish_Flood, 215 test windows
mean set size 2.535, median 3, singletons 0.0000, empty 0.0000
over the finitely thresholded classes only, mean 1.535
the true class is in its own set on 0.6651 of them

which classes appear in the sets of MQTT-DDoS-Publish_Flood windows
           class_in_set  share_of_windows  is_the_true_class  has_a_finite_threshold
       Recon-Ping_Sweep            1.0000              False                   False
 MQTT-DoS-Publish_Flood            0.7767              False                    True
MQTT-DDoS-Publish_Flood            0.6651               True                    True
          Recon-VulScan            0.0605              False                    True
               DoS-ICMP            0.0186              False                    True
                DoS-UDP            0.0140              False                    True


Now every class at once, so the three above can be placed against the rest. The table
carries the mean set size over all classes and over the finitely thresholded ones, the
median, the share of windows that came back as a single class, the empirical coverage, the
calibration count that produced the threshold, and the F1 the same model recorded on the
same test partition. Sorted by mean set size, largest first.

The comparison to make is down the last two columns. If set size is measuring difficulty
then the classes at the top of this table, the ones with the largest sets, should be the
classes with the lowest F1, and the rank correlation between the two columns should come
out clearly negative. If it comes out near zero, set size and detection difficulty are
measuring different things and the corroboration argument does not stand.


In [10]:
def per_class_summary(coverage):
    member, sizes, covered = MEMBER[coverage], SIZES[coverage], COVERED[coverage]
    rows = []
    for c, name in enumerate(CLASSES):
        mask = Y_TEST == c
        n = int(mask.sum())
        if n == 0:
            continue
        s = sizes[mask]
        rows.append(
            {
                "class": name,
                "test_n": n,
                "cal_n": int(VAL_N[c]),
                "mean_set_size": float(s.mean()),
                "mean_size_finite": float(SIZES_FINITE[coverage][mask].mean()),
                "median_set_size": float(np.median(s)),
                "share_singleton": float((s == 1).mean()),
                "share_empty": float((s == 0).mean()),
                "coverage": float(covered[mask].mean()),
                "f1": float(PER_CLASS_F1[name]) if PER_CLASS_F1 else float("nan"),
            }
        )
    table = pd.DataFrame(rows)
    for column in table.columns.drop("class"):
        if not pd.api.types.is_numeric_dtype(table[column]):
            raise TypeError(f"{column} came out as {table[column].dtype}, not numeric")
    return table.sort_values("mean_set_size", ascending=False)


SUMMARY = {coverage: per_class_summary(coverage) for coverage in COVERAGES}

print(f"every class at coverage {PRIMARY}, largest mean set size first")
print(SUMMARY[PRIMARY].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

RANK_CORR = float("nan")
if PER_CLASS_F1 is not None:
    RANK_CORR = float(SUMMARY[PRIMARY][["mean_set_size", "f1"]].corr(method="spearman").iloc[0, 1])
    print()
    print(f"rank correlation between mean set size and per-class F1: {RANK_CORR:+.4f}")
    print("  (negative means the classes with the largest sets are the classes with the "
          "lowest F1)")
    worst = SUMMARY[PRIMARY].nsmallest(4, "f1")
    print()
    print("the four lowest-F1 classes and where they sit on set size")
    print(worst[["class", "f1", "mean_set_size", "median_set_size", "share_singleton"]]
          .to_string(index=False, float_format=lambda v: f"{v:.4f}"))


every class at coverage 0.9, largest mean set size first
                  class  test_n  cal_n  mean_set_size  mean_size_finite  median_set_size  share_singleton  share_empty  coverage     f1
    MQTT-Malformed_Data      40     38         4.1500            3.1500           4.0000           0.0000       0.0000    0.9750 0.8421
       Recon-Ping_Sweep       4      2         4.0000            3.0000           4.0000           0.0000       0.0000    1.0000 0.0000
               Spoofing     104    105         3.6635            2.6635           4.0000           0.0000       0.0000    0.9519 0.7653
               DDoS-TCP    7302   8038         3.5022            2.5022           4.0000           0.0000       0.0000    0.9608 0.8137
                DoS-TCP    3282   3738         3.4951            2.4951           4.0000           0.0000       0.0000    0.8824 0.5161
          Recon-OS_Scan     123    121         3.4146            2.4146           3.0000           0.0000       0.0000    0.796

Coverage last, and reported as information rather than as a pass mark. The procedure is
built to put the true class in the set a target fraction of the time, and the fraction it
actually achieves per class is worth seeing, but nothing in this notebook rests on it. Two
classes calibrate on too few points for a guarantee to be worth stating, the calibration
and evaluation windows come from different capture sessions rather than from one
exchangeable pool, and the quantity being used from all of this is set size. A class that
undercovers is a fact about the calibration, not a failed test.

The same table at the lower target is printed underneath. Dropping the target to 0.80
lowers every threshold, which shrinks every set, and how much of the ambiguity survives
that is a check on whether the large sets are a real property of those classes or an
artefact of one particular dial setting.


In [11]:
for coverage in COVERAGES:
    table = SUMMARY[coverage]
    marginal = float(COVERED[coverage].mean())
    print(f"target {coverage}: marginal coverage over all test windows {marginal:.4f}, "
          f"mean set size {SIZES[coverage].mean():.3f}")
    print(table[["class", "cal_n", "coverage", "mean_set_size", "share_singleton", "share_empty"]]
          .sort_values("coverage")
          .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    under = table[table["coverage"] < coverage - 0.05]
    print(f"  classes more than 0.05 below the target: "
          + (", ".join(f"{r['class']} {r.coverage:.3f}" for _, r in under.iterrows())
             if len(under) else "none"))
    print()

shrink = pd.DataFrame(
    {
        "class": SUMMARY[PRIMARY]["class"],
        "mean_at_0.90": SUMMARY[PRIMARY]["mean_set_size"].to_numpy(),
        "mean_at_0.80": SUMMARY[0.80].set_index("class")
        .loc[SUMMARY[PRIMARY]["class"], "mean_set_size"]
        .to_numpy(),
    }
)
shrink["change"] = shrink["mean_at_0.80"] - shrink["mean_at_0.90"]
for column in ("mean_at_0.90", "mean_at_0.80", "change"):
    if not pd.api.types.is_numeric_dtype(shrink[column]):
        raise TypeError(f"{column} came out as {shrink[column].dtype}, not numeric")

print("what lowering the target does to the mean set size of each class")
print(shrink.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


target 0.9: marginal coverage over all test windows 0.8610, mean set size 2.895
                  class  cal_n  coverage  mean_set_size  share_singleton  share_empty
MQTT-DDoS-Publish_Flood    213    0.6651         2.5349           0.0000       0.0000
              DDoS-ICMP   6159    0.6890         3.3562           0.0000       0.0000
               DDoS-UDP   8225    0.7661         2.0539           0.0686       0.0000
               DoS-ICMP   4243    0.7805         3.2200           0.0000       0.0000
                DoS-SYN   4424    0.7940         2.5124           0.0000       0.0000
          Recon-OS_Scan    121    0.7967         3.4146           0.0000       0.0000
                DoS-TCP   3738    0.8824         3.4951           0.0000       0.0000
        Recon-Port_Scan    637    0.8840         2.8856           0.0000       0.0000
          Recon-VulScan     18    0.8889         2.8333           0.0000       0.0000
 MQTT-DoS-Connect_Flood     92    0.8936         2.8936     

The read. Three questions were asked at the top and the block below answers each of them
from the numbers just printed rather than from anything decided in advance. What it cannot
answer is whether the sets are correct in any guaranteed sense, which is not what was being
asked and is not what set sizes are for.


In [12]:
sizes = SIZES[PRIMARY]
singleton_share = float((sizes == 1).mean())
spread_all = float(
    SUMMARY[PRIMARY]["mean_size_finite"].max() - SUMMARY[PRIMARY]["mean_size_finite"].min()
)
target_row = SUMMARY[PRIMARY].set_index("class").loc[target]
target_rank = int(SUMMARY[PRIMARY]["class"].tolist().index(target)) + 1

print("=" * 78)
print(f"read, coverage {PRIMARY}, {len(sizes):,} test windows, nothing written")
print("=" * 78)
print()

print("1. are the set sizes informative")
print(f"   Across all windows the mean set size is {sizes.mean():.3f} of {N_CLASSES} classes "
      f"and {singleton_share:.1%} come back")
print(f"   as a single class. Setting aside the {len(ALWAYS_IN[PRIMARY])} class(es) admitted "
      f"to every set for want of")
print(f"   calibration points, the per-class mean ranges from "
      f"{SUMMARY[PRIMARY]['mean_size_finite'].min():.3f} to "
      f"{SUMMARY[PRIMARY]['mean_size_finite'].max():.3f}, a spread of {spread_all:.3f}.")
print("   " + ("The sizes vary across classes, so they are carrying information."
               if spread_all > 0.5 else
               "The sizes barely vary across classes, so they are not separating them."))
print()

print("2. do larger sets go with harder classes")
if PER_CLASS_F1 is not None:
    print(f"   Rank correlation between mean set size and per-class F1 is {RANK_CORR:+.4f}.")
    print("   " + ("Set size tracks measured difficulty: the ambiguous classes are the "
                   "classes the model scores worst."
                   if RANK_CORR <= -0.5 else
                   "Set size tracks measured difficulty only weakly, so the two are not "
                   "the same measurement."
                   if RANK_CORR <= -0.2 else
                   "Set size does not track measured difficulty, so it is not a proxy for it."))
else:
    print("   The per-class F1 file was not found, so this comparison did not run.")
print(f"   On the three chosen in advance, over the finitely thresholded classes: "
      f"{easy} {mean_size_finite[easy]:.3f},")
print(f"   {hard} {mean_size_finite[hard]:.3f}, {target} {mean_size_finite[target]:.3f}.")
print("   " + ("The easy class gives the smallest sets and both hard classes give larger "
               "ones, which is the expected ordering."
               if hard_larger and target_larger else
               "The expected ordering does not hold across all three."))
print()

print(f"3. does {target} come back ambiguous")
print(f"   Mean set size {target_row.mean_set_size:.3f} ({target_row.mean_size_finite:.3f} "
      f"over the finitely thresholded classes), median")
print(f"   {target_row.median_set_size:.0f}, singletons {target_row.share_singleton:.1%}, "
      f"rank {target_rank} of {N_CLASSES} by set size.")
corroborates = target_rank <= max(1, N_CLASSES // 3) and target_row.mean_size_finite > 1.5
print("   " + ("Its windows are among the most ambiguous in the dataset, which points where "
               "the separability analysis already pointed, from a method that never saw it."
               if corroborates else
               "Its windows are not among the most ambiguous, so this probe does not "
               "corroborate the separability finding for it."))
print()

print("what this does not establish")
print(f"   Coverage is reported and not claimed. {int((VAL_N < SMALL_CALIBRATION).sum())} "
      f"classes calibrate on fewer than {SMALL_CALIBRATION} windows,")
print("   the calibration and evaluation windows come from different capture sessions, and")
print("   no result from this notebook has been written to disk or to the ledger.")


read, coverage 0.9, 49,159 test windows, nothing written

1. are the set sizes informative
   Across all windows the mean set size is 2.895 of 19 classes and 0.9% come back
   as a single class. Setting aside the 1 class(es) admitted to every set for want of
   calibration points, the per-class mean ranges from 1.054 to 3.150, a spread of 2.096.
   The sizes vary across classes, so they are carrying information.

2. do larger sets go with harder classes
   Rank correlation between mean set size and per-class F1 is -0.4561.
   Set size tracks measured difficulty only weakly, so the two are not the same measurement.
   On the three chosen in advance, over the finitely thresholded classes: DDoS-SYN 1.713,
   Recon-VulScan 1.833, MQTT-DDoS-Publish_Flood 1.535.
   The expected ordering does not hold across all three.

3. does MQTT-DDoS-Publish_Flood come back ambiguous
   Mean set size 2.535 (1.535 over the finitely thresholded classes), median
   3, singletons 0.0%, rank 15 of 19 by set si